# 📚 SQL Ch.2 — Aggregation & GROUP BY
> BigQuery SQL Reference Guide, Chapter 2: COUNT · SUM/AVG/MIN/MAX · GROUP BY · HAVING · ROLLUP  
> BigQuery SQL 완전 참조 가이드 2장: COUNT · SUM/AVG/MIN/MAX · GROUP BY · HAVING · ROLLUP

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Turn many rows into summary statistics with `GROUP BY` and aggregate functions  
`GROUP BY`와 집계 함수로 여러 행을 요약 통계로 바꾼다
- [x] Explain why `HAVING` exists when `WHERE` already filters rows  
`WHERE`가 이미 행을 거르는데 왜 `HAVING`이 따로 필요한지 설명한다
- [x] Chain `WHERE → GROUP BY → HAVING → ORDER BY → LIMIT` into one BA reporting query  
`WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`를 하나의 BA 리포트 쿼리로 연결한다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** Aggregation collapses many rows down into one summary number per group — "total sales," "average salary," "how many orders." `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` compute that summary; `GROUP BY` decides *how* to split the rows into groups before summarizing (e.g. one group per region); and `HAVING` filters those *group-level* results, the same way `WHERE` filters individual rows.

**KR:** 집계는 여러 행을 그룹당 하나의 요약 숫자로 압축합니다 — "총 매출", "평균 연봉", "주문 건수" 같은 것들입니다. `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`가 그 요약값을 계산하고, `GROUP BY`는 요약하기 *전에* 행을 어떻게 그룹으로 나눌지(예: 지역별로 하나씩) 정하며, `HAVING`은 `WHERE`가 개별 행을 거르는 것과 같은 방식으로 그 *그룹 단위* 결과를 거릅니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Raw transaction tables are almost never the answer a stakeholder wants — nobody asks to see 50,000 individual order rows. They ask "what's our revenue by region?" or "which reps closed the most deals?" Aggregation is the step that turns row-level data into the summary tables and dashboards people actually read.

**KR:** 원본 트랜잭션 테이블 자체가 이해관계자가 원하는 답인 경우는 거의 없습니다 — 아무도 5만 개의 개별 주문 행을 보고 싶어하지 않습니다. "지역별 매출이 어떻게 되나요?", "어느 담당자가 가장 많이 계약했나요?"라고 묻습니다. 집계는 행 단위 데이터를 사람들이 실제로 읽는 요약 표와 대시보드로 바꿔주는 단계입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** This is arguably the single most-used chapter in BA work. Monthly revenue reports, "top N customers by spend," "which regions are underperforming" — all of it is `GROUP BY` plus an aggregate function, often with `HAVING` to focus on groups that actually matter (e.g. "only regions with 200,000+ in sales").

**KR:** BA 업무에서 아마 가장 많이 쓰이는 챕터일 것입니다. 월별 매출 리포트, "지출 상위 N개 고객", "어느 지역이 부진한가" — 모두 `GROUP BY`와 집계 함수의 조합이며, 실제로 중요한 그룹만 골라내기 위해 `HAVING`을 함께 쓰는 경우가 많습니다(예: "매출 200,000 이상인 지역만").

**Comparison / 비교표:**

| Task / 작업 | SQL | Pandas | Excel |
|---|---|---|---|
| Count rows / 행 수 | `COUNT(*)` | `len(df)` | `COUNTA` |
| Count non-null / 결측 제외 개수 | `COUNT(col)` | `df["col"].count()` | `COUNTA(range)` |
| Count unique / 고유값 개수 | `COUNT(DISTINCT col)` | `df["col"].nunique()` | Remove Duplicates + count |
| Group + summarize / 그룹별 요약 | `GROUP BY` | `df.groupby()` | PivotTable |
| Filter groups / 그룹 필터링 | `HAVING` | `.filter(lambda g: ...)` | PivotTable value filter |
| Subtotals + grand total / 소계+총계 | `ROLLUP` | `pivot_table(margins=True)` | Subtotal feature |

---
# 📝 Syntax

## Basic Syntax
`COUNT(*)` vs `COUNT(col)` vs `COUNT(DISTINCT col)` — three different questions that look similar.  
`COUNT(*)` vs `COUNT(col)` vs `COUNT(DISTINCT col)` —비슷해 보이지만 서로 다른 세 가지 질문입니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

orders = pd.DataFrame({
    "order_id":      [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id":   ["C01","C02","C01","C03","C02","C01"],
    "discount_code": ["SAVE10", None, "SAVE10", None, "WELCOME", None],
    "amount":        [45000, 32000, 61000, 28000, 95000, 15000],
})

sql = """
SELECT
    COUNT(*)                   AS total_rows,        -- every row, NULLs included / NULL 포함 전체 행
    COUNT(discount_code)       AS non_null_codes,     -- NULLs skipped / NULL 제외
    COUNT(DISTINCT discount_code) AS unique_codes,    -- distinct, NULLs skipped / 고유값, NULL 제외
    COUNT(DISTINCT customer_id)   AS unique_customers
FROM orders
"""
display(run(sql))
# Read it: 6 total rows, but only 3 have a discount_code (3 are NULL), only 2 distinct codes, 3 distinct customers.
# 읽는 법: 전체 6행이지만 discount_code가 있는 행은 3개(3개는 NULL), 고유 코드는 2개, 고유 고객은 3명.


,total_rows,non_null_codes,unique_codes,unique_customers
0,6,3,2,3


## Common Variations

In [2]:
# The most common mistake: treating COUNT(col) as if it means COUNT(*).
# 가장 흔한 실수: COUNT(col)을 COUNT(*)와 같다고 착각하는 것.
# Whenever a column can contain NULL, compare the two to see how much data is missing.
# 열에 NULL이 있을 수 있다면 두 값을 비교해서 결측치 규모를 파악하세요.
display(run("SELECT COUNT(*) AS all_rows, COUNT(discount_code) AS has_code FROM orders"))


,all_rows,has_code
0,6,3


---
# 🧪 Small Examples

## Example 1 — SUM / AVG / MIN / MAX
**EN:** These four aggregate functions summarize a numeric column across all matched rows. Like `COUNT(col)`, they all automatically skip `NULL` values — they operate only on the valid entries, not the full row count.  
**KR:** 이 네 집계 함수는 매칭된 모든 행에 걸쳐 숫자 열을 요약합니다. `COUNT(col)`처럼 `NULL` 값을 자동으로 건너뛰며, 전체 행 수가 아니라 유효한 값들만 대상으로 계산합니다.

In [3]:
monthly_sales = pd.DataFrame({
    "month": ["Jan","Feb","Mar","Apr","May"],
    "sales": [5200000, 4800000, 6100000, 5500000, 4300000],
})

sql = """
SELECT
    SUM(sales) AS total_sales,
    AVG(sales) AS avg_sales,
    MIN(sales) AS min_sales,
    MAX(sales) AS max_sales
FROM monthly_sales
"""
display(run(sql))


,total_sales,avg_sales,min_sales,max_sales
0,25900000.0,5180000.0,4300000,6100000


## Example 2 — GROUP BY: Single & Multiple Columns / 단일 열, 복수 열
**EN:** `GROUP BY` splits rows into buckets before aggregating — one output row per unique value (or combination of values).  
**The rule:** every column in `SELECT` that isn't wrapped in an aggregate function must appear in `GROUP BY`, or the query is invalid.  
**KR:** `GROUP BY`는 집계하기 전에 행을 그룹으로 나눕니다 — 고유값(또는 값의 조합)마다 결과 행이 하나씩 나옵니다.  
**규칙:** `SELECT`에 나열한 열 중 집계 함수로 감싸지 않은 열은 반드시 `GROUP BY`에도 있어야 하며, 그러지 않으면 쿼리가 무효입니다.

In [4]:
sales = pd.DataFrame({
    "region":   ["서울","서울","부산","부산","인천","서울"],
    "category": ["전자","의류","전자","의류","전자","전자"],
    "amount":   [320000, 85000, 150000, 95000, 67000, 190000],
})

print("-- single-column GROUP BY / 단일 열 GROUP BY --")
sql1 = """
SELECT
    region,
    SUM(amount) AS total,
    COUNT(*) AS order_count,
    ROUND(AVG(amount), 1) AS avg_amount
FROM sales
GROUP BY region
"""
display(run(sql1))

print("-- multi-column GROUP BY: one row per (region, category) combo --")
print("-- 복수 열 GROUP BY: (region, category) 조합마다 한 행 --")
sql2 = """
SELECT region, category, SUM(amount) AS total
FROM sales
GROUP BY region, category
"""
display(run(sql2))


-- single-column GROUP BY / 단일 열 GROUP BY --


,region,total,order_count,avg_amount
0,서울,595000.0,3,198333.3
1,부산,245000.0,2,122500.0
2,인천,67000.0,1,67000.0


-- multi-column GROUP BY: one row per (region, category) combo --
-- 복수 열 GROUP BY: (region, category) 조합마다 한 행 --


,region,category,total
0,서울,전자,510000.0
1,부산,전자,150000.0
2,부산,의류,95000.0
3,인천,전자,67000.0
4,서울,의류,85000.0


## Example 3 — HAVING: Filtering Aggregated Results / 집계 결과 필터링
**EN:** `HAVING` filters *after* grouping, using the aggregate value itself (`SUM(amount) >= 200000`) — something `WHERE` cannot do, since `WHERE` only sees individual rows, before any grouping happens.  
**KR:** `HAVING`은 집계값 자체(`SUM(amount) >= 200000`)를 기준으로 그룹화 *이후*에 필터링합니다 — `WHERE`는 그룹화가 일어나기 전 개별 행만 보므로 이건 `WHERE`가 할 수 없는 일입니다.

In [5]:
sql = """
SELECT region, SUM(amount) AS total
FROM sales
GROUP BY region
HAVING SUM(amount) >= 200000
"""
display(run(sql))
# 인천 (67,000) is excluded -- didn't meet the 200,000 threshold.
# 인천(67,000)은 20만 기준을 못 채워서 제외됨.

# ⚠️ BigQuery allows reusing a SELECT alias in HAVING (e.g. "HAVING total >= 200000"),
# but some other dialects (older MySQL) don't. Repeating SUM(amount) is the safe, portable choice.
# ⚠️ BigQuery는 HAVING에서 SELECT 별칭 재사용을 허용하지만(예: "HAVING total >= 200000"),
# 일부 다른 dialect(구버전 MySQL)는 그렇지 않습니다. SUM(amount)를 그대로 반복하는 것이 이식성 있는 선택입니다.


,region,total
0,부산,245000.0
1,서울,595000.0


## Example 4 — HAVING vs WHERE: Understanding Execution Order / 실행 순서로 이해하기
**EN:** SQL's real execution order is `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT` — completely different from the order you type it in. `WHERE` removes individual rows *before* grouping; `HAVING` removes whole groups *after* grouping. This single fact explains almost every "why doesn't this work" moment in SQL.  
**KR:** SQL의 실제 실행 순서는 `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`이며, 작성한 순서와는 완전히 다릅니다. `WHERE`는 그룹화 *전*에 개별 행을 제거하고, `HAVING`은 그룹화 *후*에 그룹 전체를 제거합니다. 이 사실 하나가 SQL에서 "왜 이게 안 되지?" 싶은 순간의 대부분을 설명해줍니다.

| Step / 순서 | Clause / 절 | Role / 역할 |
|---|---|---|
| 1 | `FROM` | Pick the table (incl. JOIN) / 테이블 선택 |
| 2 | `WHERE` | Filter rows, *before* grouping / 그룹화 **이전** 행 필터링 |
| 3 | `GROUP BY` | Form groups / 그룹 묶기 |
| 4 | `HAVING` | Filter groups, *after* grouping / 그룹화 **이후** 결과 필터링 |
| 5 | `SELECT` | Pick columns, compute aliases / 열 선택, 별칭 계산 |
| 6 | `ORDER BY` | Sort / 정렬 |
| 7 | `LIMIT` | Cap row count / 행 수 제한 |

In [6]:
orders4 = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": ["C01","C02","C01","C03","C02","C01"],
    "region":      ["서울","부산","서울","인천","부산","서울"],
    "status":      ["완료","완료","취소","완료","완료","완료"],
    "amount":      [45000, 32000, 61000, 28000, 95000, 15000],
})

sql = """
SELECT
    region,
    SUM(amount) AS total
FROM orders4
WHERE status = '완료'          -- ① first: drop the cancelled row (1003) / 먼저 취소(1003) 행 제거
GROUP BY region                -- ② then: form groups / 그룹화
HAVING SUM(amount) >= 50000    -- ③ finally: drop groups under 50,000 / 그룹 합계 5만 미만 제외
"""
display(run(sql))
# Step-by-step / 단계별 추적:
#  1) WHERE  -> removes order 1003 (cancelled, 61000). 5 rows remain.  / 1003(취소) 제거, 5건 남음
#  2) GROUP BY region -> 서울: 45000+15000=60000 / 부산: 32000+95000=127000 / 인천: 28000
#  3) HAVING -> 인천(28000) excluded, under the 50000 threshold / 인천 제외


,region,total
0,부산,127000.0
1,서울,60000.0


## Example 5 — GROUP BY + ORDER BY Combination / GROUP BY + ORDER BY 조합
**EN:** `ORDER BY` runs *after* `GROUP BY` in the execution order (step 6, vs step 3) — so unlike `HAVING`, it can *always* safely reuse a `SELECT` alias, in every SQL dialect.  
**KR:** `ORDER BY`는 실행 순서상 `GROUP BY`보다 나중(3단계가 아니라 6단계)에 실행되므로, `HAVING`과 달리 모든 SQL dialect에서 `SELECT` 별칭을 *항상* 안전하게 재사용할 수 있습니다.

In [7]:
sql = """
SELECT region, SUM(amount) AS total
FROM sales
GROUP BY region
ORDER BY total DESC     -- ✅ reusing the alias defined in SELECT is always safe here
"""
display(run(sql))


,region,total
0,서울,595000.0
1,부산,245000.0
2,인천,67000.0


## Example 6 — ROLLUP: Automatic Subtotal & Grand-Total Rows / 소계·합계 행 자동 추가
**EN:** `GROUP BY ROLLUP(a, b)` adds automatic subtotal rows (one `NULL` for `b`, meaning "totalled up to this level of `a`") and a final grand-total row (`NULL` for both) — like an Excel PivotTable's Subtotal feature. `GROUPING(col)` returns `1` on those synthetic rows and `0` on real detail rows, so you can tell a "real NULL" apart from a "rollup NULL."  
**KR:** `GROUP BY ROLLUP(a, b)`는 자동으로 소계 행(`b`가 `NULL`인 행 — "`a` 레벨까지 집계됨"이라는 뜻)과 마지막 총계 행(둘 다 `NULL`)을 추가합니다 — 엑셀 피벗테이블의 부분합 기능과 비슷합니다. `GROUPING(col)`은 이런 합성 행에서는 `1`, 실제 상세 행에서는 `0`을 반환하므로 "진짜 NULL"과 "롤업으로 생긴 NULL"을 구분할 수 있습니다.

⚠️ **EN:** Row order after `ROLLUP` isn't guaranteed — add an explicit `ORDER BY` (as below) if you need detail rows, subtotals, and the grand total to appear in a predictable sequence.  
⚠️ **KR:** `ROLLUP` 이후 행의 순서는 보장되지 않습니다 — 상세 행, 소계, 총계가 예측 가능한 순서로 나오게 하려면 아래처럼 명시적으로 `ORDER BY`를 추가하세요.

In [8]:
sales_r = pd.DataFrame({
    "region":   ["서울","서울","부산","부산"],
    "category": ["전자","의류","전자","의류"],
    "amount":   [300000, 100000, 150000, 50000],
})

print("-- ROLLUP: detail rows + region subtotal + grand total --")
print("-- ROLLUP: 상세 행 + 지역별 소계 + 전체 합계 --")
sql = """
SELECT
    region,
    category,
    SUM(amount) AS total,
    GROUPING(region)   AS is_region_total,    -- 1 = this row is a subtotal/grand-total for region
    GROUPING(category) AS is_category_total   -- 1 = this row is a subtotal/grand-total for category
FROM sales_r
GROUP BY ROLLUP(region, category)
ORDER BY GROUPING(region), region, GROUPING(category), category
"""
display(run(sql))
# Read it: category=NULL, is_category_total=1 rows are per-region subtotals;
#          region=NULL, is_region_total=1 row is the grand total (last row).
# 읽는 법: category=NULL, is_category_total=1 행은 지역별 소계;
#          region=NULL, is_region_total=1 행이 전체 합계(마지막 행).


-- ROLLUP: detail rows + region subtotal + grand total --
-- ROLLUP: 상세 행 + 지역별 소계 + 전체 합계 --


,region,category,total,is_region_total,is_category_total
0,부산,의류,50000.0,0,0
1,부산,전자,150000.0,0,0
2,부산,NaN,200000.0,0,1
3,서울,의류,100000.0,0,0
4,서울,전자,300000.0,0,0
5,서울,NaN,400000.0,0,1
6,NaN,NaN,600000.0,1,1


## Example 7 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** chains all five clauses from Example 4's table into the single most common BA reporting query: filter → group → filter groups → sort → cap. **Pattern B** compares `COUNT(*)` against `COUNT(DISTINCT customer_id)` per group to spot repeat-purchase behavior — when the two numbers differ, some customers ordered more than once.  
**KR:** **패턴 A**는 예제 4의 다섯 절을 모두 이어 붙여, BA가 가장 자주 쓰는 리포트 쿼리 하나를 완성합니다: 필터 → 그룹화 → 그룹 필터 → 정렬 → 개수 제한. **패턴 B**는 그룹별로 `COUNT(*)`와 `COUNT(DISTINCT customer_id)`를 비교해 재구매 신호를 찾습니다 — 두 값이 다르면 어떤 고객이 두 번 이상 주문했다는 뜻입니다.

In [9]:
print("-- Pattern A: WHERE + GROUP BY + HAVING + ORDER BY + LIMIT --")
sql_a = """
SELECT region, SUM(amount) AS total
FROM orders4
WHERE status = '완료'
GROUP BY region
HAVING SUM(amount) >= 50000
ORDER BY total DESC
LIMIT 2
"""
display(run(sql_a))

print("-- Pattern B: COUNT(*) vs COUNT(DISTINCT) -- spotting repeat purchases --")
print("-- 패턴 B: COUNT(*) vs COUNT(DISTINCT) -- 재구매 신호 찾기 --")
orders_b = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005, 1006],
    "customer_id": ["C01","C02","C01","C03","C04","C05"],
    "region":      ["서울","서울","서울","부산","부산","인천"],
})
sql_b = """
SELECT
    region,
    COUNT(*) AS order_count,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM orders_b
GROUP BY region
"""
display(run(sql_b))
# 서울: order_count(3) > unique_customers(2) -> customer C01 ordered twice = repeat purchase.
# 부산/인천: the two numbers match -> every order was a different, first-time customer.
# 서울: order_count(3) > unique_customers(2) -> C01 고객이 2번 주문 = 재구매 발생.
# 부산/인천: 두 값이 같음 -> 모든 주문이 서로 다른, 처음 구매하는 고객.


-- Pattern A: WHERE + GROUP BY + HAVING + ORDER BY + LIMIT --


,region,total
0,부산,127000.0
1,서울,60000.0


-- Pattern B: COUNT(*) vs COUNT(DISTINCT) -- spotting repeat purchases --
-- 패턴 B: COUNT(*) vs COUNT(DISTINCT) -- 재구매 신호 찾기 --


,region,order_count,unique_customers
0,서울,3,2
1,부산,2,2
2,인천,1,1


## Example 8 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `AVG` `COUNT` `HAVING` `DESC` `ROLLUP`  
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `AVG` `COUNT` `HAVING` `DESC` `ROLLUP`

In [12]:
employees_p = pd.DataFrame({
    "emp_id": ["E01","E02","E03","E04","E05","E06","E07"],
    "name":   ["김민수","이영희","박준호","최서연","정대현","윤소영","한지훈"],
    "dept":   ["영업","마케팅","개발","영업","마케팅","개발","영업"],
    "salary": [4200000, 3800000, 5100000, 3500000, 4500000, 5800000, 4000000],
    "bonus":  [200000, None, 300000, None, 150000, 400000, 100000],
})

# Q1. Per-dept average salary, headcount, and how many got a bonus.
# Q1. 부서별 평균 연봉, 전체 인원 수, 보너스를 받은 인원 수.
q1 = """
SELECT
    dept,
    AVG(salary) AS avg_salary,
    COUNT(*) AS headcount,
    COUNT(bonus) AS bonus_count
FROM employees_p
GROUP BY dept
"""
display(run(q1))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q2. Only departments with avg salary >= 4,000,000, highest average first.
# Q2. 평균 연봉이 4,000,000 이상인 부서만, 평균 연봉 내림차순으로.
q2 = """
SELECT dept, AVG(salary) AS avg_salary
FROM employees_p
GROUP BY dept
HAVING AVG(salary) >= 4000000
ORDER BY avg_salary DESC
"""
display(run(q2))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q3. Headcount per dept, plus a grand-total row, using ROLLUP.
# Q3. ROLLUP을 사용해 부서별 인원 수와 전체 인원 수(합계 행)를 함께.
q3 = """
SELECT dept, COUNT(*) AS headcount
FROM employees_p
GROUP BY ROLLUP(dept)
"""
display(run(q3))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,dept,avg_salary,headcount,bonus_count
0,개발,5450000.0,2,2
1,마케팅,4150000.0,2,1
2,영업,3900000.0,3,2


,dept,avg_salary
0,개발,5450000.0
1,마케팅,4150000.0


,dept,headcount
0,NaN,7
1,개발,2
2,마케팅,2
3,영업,3


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT
    dept,
    AVG(salary) AS avg_salary,
    COUNT(*) AS headcount,
    COUNT(bonus) AS bonus_count
FROM employees_p
GROUP BY dept

-- Q2
SELECT dept, AVG(salary) AS avg_salary
FROM employees_p
GROUP BY dept
HAVING AVG(salary) >= 4000000
ORDER BY avg_salary DESC

-- Q3
SELECT dept, COUNT(*) AS headcount
FROM employees_p
GROUP BY ROLLUP(dept)
```
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Assuming `COUNT(discount_code)` equals `COUNT(*)`**
- EN: `COUNT(col)` silently skips `NULL` values, while `COUNT(*)` counts every row regardless. If a column has missing data, these two numbers will differ — and that gap *is* your missing-data count.
- KR: `COUNT(col)`은 `NULL` 값을 조용히 건너뛰지만, `COUNT(*)`는 무조건 모든 행을 셉니다. 열에 결측 데이터가 있다면 두 숫자는 달라지며, 그 차이가 곧 결측치 개수입니다.
- ✅ Fix / 해결법: Whenever `NULL` is possible, compare `COUNT(*)` and `COUNT(col)` side by side to size up missing data before trusting an aggregate.   `NULL` 가능성이 있다면 항상 `COUNT(*)`와 `COUNT(col)`을 나란히 비교해서 집계를 신뢰하기 전에 결측치 규모를 파악하세요.

**Mistake 2 — Putting a group-level condition in `WHERE` instead of `HAVING`**
- EN: `WHERE SUM(amount) >= 200000` fails, because `WHERE` runs *before* grouping and aggregation even happen — there's no `SUM` yet for it to compare against.
- KR: `WHERE SUM(amount) >= 200000`은 실패합니다. `WHERE`는 그룹화와 집계가 일어나기 *전에* 실행되므로, 비교할 `SUM` 자체가 아직 존재하지 않기 때문입니다.
- ✅ Fix / 해결법: Use `HAVING SUM(amount) >= 200000` for conditions on aggregated values; keep `WHERE` for conditions on raw row values.  
집계값에 대한 조건은 `HAVING`을, 원본 행 값에 대한 조건은 `WHERE`를 사용하세요.

**Mistake 3 — Selecting a non-aggregated column that's missing from GROUP BY**
- EN: `SELECT region, category, SUM(amount) FROM sales GROUP BY region` errors, because `category` is neither aggregated nor listed in `GROUP BY` — the database doesn't know which of the (possibly several) category values to show per region.
- KR: `SELECT region, category, SUM(amount) FROM sales GROUP BY region`는 오류가 납니다. `category`가 집계되지도, `GROUP BY`에 들어있지도 않아서, 지역당 여러 개 있을 수 있는 category 값 중 어떤 걸 보여줘야 할지 알 수 없기 때문입니다.
- ✅ Fix / 해결법: Either add `category` to `GROUP BY`, or wrap it in an aggregate like `MAX(category)`.  
`GROUP BY`에 `category`를 추가하거나, `MAX(category)`처럼 집계 함수로 감싸세요.

**Mistake 4 — Treating ROLLUP's `NULL` as missing data**
- EN: A `NULL` produced by `ROLLUP` doesn't mean "unknown" — it means "totalled up to this level." Filtering it out with `WHERE region IS NOT NULL` (applied *before* the rollup) would remove real detail rows too if you're not careful about clause order.
- KR: `ROLLUP`이 만든 `NULL`은 "값을 모름"이 아니라 "이 레벨까지 집계됨"을 뜻합니다. 절 순서를 주의하지 않고 `WHERE region IS NOT NULL`로 걸러내려 하면(롤업 *이전*에 적용되므로) 진짜 상세 행까지 함께 사라질 수 있습니다.
- ✅ Fix / 해결법: Use `GROUPING(col) = 1` to identify synthetic subtotal/grand-total rows instead of testing the column for `NULL`.  
열이 `NULL`인지 검사하는 대신 `GROUPING(col) = 1`로 합성된 소계/총계 행을 구분하세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Memorize the seven-step order once — `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT` — and `WHERE` vs `HAVING` stops being a rule you memorize and becomes a rule you *derive*.  
 7단계 순서 `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`를 한 번 외워두면, `WHERE`와 `HAVING`의 차이는 암기할 규칙이 아니라 스스로 *유도해낼 수 있는* 규칙이 됩니다.
- `COUNT(*)` vs `COUNT(col)` is a fast, free data-quality check — run it on any new table before you trust it.  
 `COUNT(*)` vs `COUNT(col)`은 빠르고 공짜인 데이터 품질 점검입니다 — 새로운 테이블을 신뢰하기 전에 항상 먼저 돌려보세요.
- `ROUND(AVG(...), 1)` is a habit worth building now — raw averages routinely come back with long decimal tails that look sloppy in a report.  
 `ROUND(AVG(...), 1)`은 지금부터 들여야 할 습관입니다 — 평균값은 리포트에서 지저분해 보이는 긴 소수점을 달고 나오는 경우가 흔합니다.
- `ROLLUP` is the SQL-native version of an Excel PivotTable's "Subtotal" checkbox — reach for it whenever a stakeholder wants subtotal rows baked into the output.  
 `ROLLUP`은 엑셀 피벗테이블의 "부분합" 체크박스에 해당하는 SQL 기능입니다 — 이해관계자가 결과에 소계 행이 포함되길 원할 때 바로 이걸 떠올리세요.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics
 2. Aggregation & GROUP BY       ← ★ YOU ARE HERE / 지금 여기
 3. JOIN
 4. Subquery & CTE
 5. Conditions & NULL Handling
 6. String & Date Functions
 7. Window Functions
 8. BA-Specific Patterns
```

```
Full query execution order / 전체 쿼리 실행 순서
──────────────────────────────────────────────
 FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
 (Ch.1 taught FROM/WHERE/SELECT/ORDER BY/LIMIT; Ch.2 adds GROUP BY + HAVING)
 (1장은 FROM/WHERE/SELECT/ORDER BY/LIMIT을, 2장은 GROUP BY + HAVING을 추가)
```

*How is today's topic connected to other concepts?*

**EN:** Chapter 2 doesn't replace anything from Chapter 1 — it *inserts* two new steps (`GROUP BY`, `HAVING`) into the same execution pipeline you already learned, which is why `WHERE`, `ORDER BY`, and `LIMIT` all still work exactly as before. Looking ahead, Chapter 3 (`JOIN`) will combine tables *before* this pipeline even starts (`JOIN` happens inside `FROM`), so "revenue by region" often becomes "revenue by region, where region lives in a different table" — the aggregation logic here doesn't change at all.

**KR:** 2장은 1장의 내용을 대체하지 않습니다 — 이미 배운 실행 파이프라인에 새 단계 두 개(`GROUP BY`, `HAVING`)를 *끼워 넣는* 것뿐이라, `WHERE`·`ORDER BY`·`LIMIT`은 이전과 똑같이 동작합니다. 앞으로 배울 3장(`JOIN`)은 이 파이프라인이 시작되기도 *전에*(`JOIN`은 `FROM` 안에서 일어남) 테이블을 결합하므로, "지역별 매출"이 종종 "다른 테이블에 있는 지역 기준 매출"이 되는데, 여기서 배운 집계 로직 자체는 전혀 바뀌지 않습니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** Finance asks: *"For completed orders only, which regions brought in at least ₩50,000 total this period? Rank them highest first."* This is Example 7's Pattern A, applied end to end.
**KR:** 재무팀이 묻습니다: *"완료된 주문만 기준으로, 이번 기간에 5만원 이상 매출을 낸 지역이 어디야? 큰 순서대로 정렬해줘."* 예제 7의 패턴 A를 그대로 적용하면 됩니다.

**To-do / 할 일:**
- [x] Keep only `완료` (completed) orders  
`완료` 상태인 주문만 남긴다
- [x] Group by region and total the amount  
지역별로 그룹화하고 금액을 합산한다
- [x] Drop any region under the ₩50,000 threshold  
5만원 기준 미달 지역은 제외한다
- [x] Sort so the biggest region is first  
가장 큰 지역이 먼저 오도록 정렬한다

In [11]:
orders_biz = pd.DataFrame({
    "order_id":    [2001, 2002, 2003, 2004, 2005, 2006, 2007],
    "customer_id": ["C01","C02","C03","C01","C04","C02","C05"],
    "region":      ["서울","부산","인천","서울","대구","부산","인천"],
    "status":      ["완료","완료","취소","완료","완료","완료","완료"],
    "amount":      [30000, 45000, 90000, 22000, 12000, 41000, 18000],
})

sql = """
SELECT region, SUM(amount) AS total_revenue
FROM orders_biz
WHERE status = '완료'
GROUP BY region
HAVING SUM(amount) >= 50000
ORDER BY total_revenue DESC
"""
display(run(sql))


,region,total_revenue
0,부산,86000.0
1,서울,52000.0


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` collapse rows into summary numbers, and they all skip `NULL` automatically — which is why `COUNT(*)` and `COUNT(col)` can disagree. `GROUP BY` decides how rows get bucketed before that summarizing happens, and every non-aggregated `SELECT` column must appear in it. `HAVING` filters those group-level results, while `WHERE` still filters individual rows before grouping — the real execution order `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT` is what makes that distinction make sense. `ROLLUP` layers automatic subtotal and grand-total rows on top of a normal `GROUP BY`, using `NULL` to mark "totalled at this level," which `GROUPING()` can detect explicitly.

**KR:** `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`는 행을 요약 숫자로 압축하며, 모두 `NULL`을 자동으로 건너뜁니다 — 그래서 `COUNT(*)`와 `COUNT(col)`이 다를 수 있습니다. `GROUP BY`는 그 요약이 일어나기 전에 행을 어떻게 묶을지 정하며, 집계되지 않은 `SELECT` 열은 모두 여기 들어가야 합니다. `HAVING`은 그 그룹 단위 결과를 필터링하고, `WHERE`는 여전히 그룹화 전 개별 행을 필터링합니다 — 실제 실행 순서 `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`가 이 구분을 이해되게 만들어줍니다. `ROLLUP`은 일반 `GROUP BY` 위에 자동 소계·총계 행을 얹으며, "이 레벨까지 집계됨"을 `NULL`로 표시하고, `GROUPING()`으로 명시적으로 구분할 수 있습니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** Aggregation is "group first, then summarize, then optionally filter the summary" — and the moment you internalize that `WHERE` filters *rows before* grouping while `HAVING` filters *groups after*, the whole chapter becomes one coherent idea instead of a list of separate keywords.

> **KR:** 집계는 "먼저 그룹으로 나누고, 요약하고, 필요하면 그 요약을 다시 거른다"는 것이며, `WHERE`는 그룹화 *전* 행을, `HAVING`은 그룹화 *후* 그룹을 거른다는 것을 체득하는 순간, 이번 챕터 전체가 따로 외울 키워드 목록이 아니라 하나로 맞물린 개념이 됩니다.

---
# ❓ Review Questions

**Q1.** Why can `COUNT(*)` and `COUNT(discount_code)` return different numbers on the exact same table?  
**Q1.** 같은 테이블인데 `COUNT(*)`와 `COUNT(discount_code)`가 왜 서로 다른 숫자를 반환할 수 있는가?

COUNT(*) counts every row; COUNT(col) skips NULLs, so they differ when the column has missing values.  
같은 테이블이어도 결측이 있으면 두 숫자가 갈립니다.

**Q2.** `SELECT region, category, SUM(amount) FROM sales GROUP BY region` throws an error. What's missing, and why does SQL require it?  
**Q2.** `SELECT region, category, SUM(amount) FROM sales GROUP BY region`는 오류가 난다. 무엇이 빠졌고, SQL이 왜 이걸 요구하는가?

Every non-aggregated SELECT column must appear in GROUP BY, or SQL cannot pick one value per group.  
빠진 것은 category입니다. 그룹당 값이 하나여야 해서 요구합니다.

**Q3.** In your own words, why can't `WHERE` filter on `SUM(amount) >= 200000`, but `HAVING` can?  
**Q3.** 왜 `WHERE`는 `SUM(amount) >= 200000` 조건으로 필터링할 수 없고, `HAVING`은 할 수 있는지 스스로의 말로 설명해보라.

WHERE filters rows before grouping, so SUM() does not exist yet. HAVING filters groups after aggregation, so it can.  
행 필터는 WHERE, 그룹 합계 필터는 HAVING입니다.

**Q4.** Walk through the 7-step execution order for: `SELECT region, SUM(amount) AS total FROM orders WHERE status='완료' GROUP BY region HAVING SUM(amount)>=50000 ORDER BY total DESC LIMIT 2` — which step runs first, and which runs last?  
**Q4.** `SELECT region, SUM(amount) AS total FROM orders WHERE status='완료' GROUP BY region HAVING SUM(amount)>=50000 ORDER BY total DESC LIMIT 2`의 7단계 실행 순서를 따라가 보라 — 어떤 단계가 가장 먼저, 어떤 단계가 가장 나중에 실행되는가?

First: FROM. Last: LIMIT. Full order is FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT.  
작성 순서와 다릅니다. 먼저 테이블, 마지막이 개수 제한입니다.

**Q5.** After `GROUP BY ROLLUP(region, category)`, a row shows `region='서울', category=NULL`. What does that `NULL` actually mean, and how would `GROUPING(category)` confirm it?  
**Q5.** `GROUP BY ROLLUP(region, category)` 이후 한 행이 `region='서울', category=NULL`로 나왔다. 이 `NULL`은 실제로 무엇을 의미하며, `GROUPING(category)`로 이를 어떻게 확인할 수 있는가?

That NULL means “totalled across all categories for Seoul,” not missing data. GROUPING(category) = 1 confirms it is a ROLLUP subtotal row.  
서울 합계 행입니다. GROUPING(category)=1이면 소계, 0이면 진짜 결측입니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*